# NanoGPT Japanese + TurboQuant

**nanoGPT + TurboQuant KV Cache Compression on Japanese Text**

This notebook demonstrates:
1. Downloading and preparing Japanese text from Aozora Bunko (青空文庫)
2. Character-level Japanese tokenizer
3. Training a small GPT on classic Japanese literature (block_size=1024)
4. Inference with stop sequences (。sentence boundaries)
5. Standard vs TurboQuant comparison at longer context

**Training corpus:** Works by 宮沢賢治, 芥川龍之介, 太宰治, 夏目漱石 from 青空文庫

**References:**
- [nanoGPT](https://github.com/karpathy/nanoGPT) - Andrey Karpathy
- [TurboQuant](https://arxiv.org/abs/2504.19874) - ICLR 2026
- [青空文庫](https://www.aozora.gr.jp/) - Public domain Japanese literature

## 0. Setup

In [ ]:
import os, sys

# Clone the repository
REPO_DIR = 'NanoGPT_with_TurboQuant'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/3t14/NanoGPT_with_TurboQuant.git

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print(f'Working directory: {os.getcwd()}')

!pip install requests numpy -q

In [ ]:
import torch
import numpy as np
import math
import time
import pickle
import re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB')

## 1. Download & Prepare Japanese Text

Download classic Japanese literary works from Aozora Bunko (青空文庫).
Multiple authors are included to increase corpus diversity and reduce memorization.

| Author | Works |
|--------|-------|
| 宮沢賢治 | 銀河鉄道の夜, セロ弾きのゴーシュ, やまなし, 注文の多い料理店, よだかの星 |
| 芥川龍之介 | 羅生門, 蜘蛛の糸, 杜子春 |
| 太宰治 | 走れメロス |
| 夏目漱石 | 夢十夜 |

In [ ]:
import requests
import zipfile
import io
import glob

def download_aozora(card_id, file_id, ruby_id):
    """Download a work from Aozora Bunko and return cleaned text."""
    url = f'https://www.aozora.gr.jp/cards/{card_id}/files/{file_id}_ruby_{ruby_id}.zip'
    print(f'Downloading: {url}')
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f'  Failed ({resp.status_code}), trying alternate URL...')
        url = f'https://www.aozora.gr.jp/cards/{card_id}/files/{file_id}_{ruby_id}.zip'
        resp = requests.get(url)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for name in zf.namelist():
            if name.endswith('.txt'):
                raw = zf.read(name)
                for enc in ['shift_jis', 'utf-8', 'cp932']:
                    try:
                        text = raw.decode(enc)
                        print(f'  Decoded with {enc}: {len(text)} chars')
                        return text
                    except (UnicodeDecodeError, LookupError):
                        continue
    raise RuntimeError('Could not decode text')

def clean_aozora(text):
    """Clean Aozora Bunko formatting."""
    lines = text.split('\n')
    start = 0
    for i, line in enumerate(lines):
        if '---' in line and i > 5:
            start = i + 1
            break
    end = len(lines)
    for i in range(len(lines) - 1, 0, -1):
        if lines[i].startswith('底本') or lines[i].startswith('青空文庫'):
            end = i
            break
    text = '\n'.join(lines[start:end])
    text = re.sub(r'《[^》]*》', '', text)
    text = text.replace('｜', '')
    text = re.sub(r'［＃[^］]*］', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# Download works from multiple authors
# IDs verified from Aozora Bunko official pages and GitHub mirror
works = [
    # 宮沢賢治 (card: 000081)
    ('000081', '456',   '145',   '銀河鉄道の夜'),
    ('000081', '470',   '3987',  'セロ弾きのゴーシュ'),
    ('000081', '472',   '654',   'やまなし'),
    ('000081', '43754', '17594', '注文の多い料理店'),
    ('000081', '473',   '467',   'よだかの星'),
    # 芥川龍之介 (card: 000879)
    ('000879', '127',   '150',   '羅生門'),
    ('000879', '92',    '164',   '蜘蛛の糸'),
    ('000879', '43015', '17393', '杜子春'),
    # 太宰治 (card: 000035)
    ('000035', '1567',  '4948',  '走れメロス'),
    # 夏目漱石 (card: 000148)
    ('000148', '799',   '6024',  '夢十夜'),
]

all_text = ''
for card_id, file_id, ruby_id, title in works:
    try:
        raw = download_aozora(card_id, file_id, ruby_id)
        cleaned = clean_aozora(raw)
        print(f'  {title}: {len(cleaned)} chars (cleaned)')
        all_text += f'\n\n{cleaned}'
    except Exception as e:
        print(f'  {title}: FAILED - {e}')

all_text = all_text.strip()
print(f'\nTotal corpus: {len(all_text):,} characters')
print(f'Works downloaded: {len([w for w in works])}')

In [ ]:
# Preview the text
print('=== First 500 characters ===')
print(all_text[:500])
print('\n=== Last 300 characters ===')
print(all_text[-300:])

## 2. Build Character-Level Japanese Tokenizer

Same approach as nanoGPT's Shakespeare tokenizer, but for Japanese characters.
A single novel typically uses 2000-3000 unique characters.

In [ ]:
# Build character-level tokenizer
chars = sorted(list(set(all_text)))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
print(f'Sample characters: {chars[:30]}...')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# Verify roundtrip
test = '銀河鉄道の夜'
assert decode(encode(test)) == test
print(f'Encode/decode test passed: "{test}" -> {encode(test)} -> "{decode(encode(test))}"')

# Create train/val split
n = len(all_text)
train_data = all_text[:int(n * 0.9)]
val_data = all_text[int(n * 0.9):]

train_ids = np.array(encode(train_data), dtype=np.uint16)
val_ids = np.array(encode(val_data), dtype=np.uint16)

print(f'Train: {len(train_ids):,} tokens')
print(f'Val: {len(val_ids):,} tokens')

# Save to data/aozora/
data_dir = os.path.join('data', 'aozora')
os.makedirs(data_dir, exist_ok=True)

train_ids.tofile(os.path.join(data_dir, 'train.bin'))
val_ids.tofile(os.path.join(data_dir, 'val.bin'))

meta = {'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}
with open(os.path.join(data_dir, 'meta.pkl'), 'wb') as f:
    pickle.dump(meta, f)

print(f'\nSaved to {data_dir}/')

## 3. Train on Japanese Text

Train a small GPT with **block_size=1024** (longer context for Japanese prose).
Higher dropout (0.35) prevents memorization while keeping sufficient training iterations.

| Parameter | Value | Note |
|-----------|-------|------|
| block_size | 1024 | 4x longer than Shakespeare demo |
| batch_size | 16 | Reduced for memory with longer context |
| gradient_accumulation_steps | 4 | Effective batch = 64 |
| n_layer / n_head / n_embd | 6 / 6 / 384 | Same small model |
| dropout | 0.35 | Higher to prevent memorization |
| max_iters | 3000 | Sufficient training with diverse corpus |

In [ ]:
# Train with block_size=1024
# dropout=0.35 to prevent memorization, corpus is now ~2x larger with diverse authors
# batch_size=16 to fit in T4 GPU memory with longer context
!python train.py \
    --dataset=aozora \
    --out_dir=out-aozora \
    --device={device} \
    --compile=False \
    --block_size=1024 \
    --batch_size=16 \
    --gradient_accumulation_steps=4 \
    --n_layer=6 \
    --n_head=6 \
    --n_embd=384 \
    --dropout=0.35 \
    --learning_rate=1e-3 \
    --max_iters=3000 \
    --eval_interval=300 \
    --eval_iters=100 \
    --lr_decay_iters=3000 \
    --min_lr=1e-4 \
    --warmup_iters=100 \
    --always_save_checkpoint=True \
    --use_turboquant=True \
    --turboquant_bits=3

## 4. Generate Japanese Text with Stop Sequences

Generate text that stops at natural Japanese sentence boundaries (。) or paragraph breaks (\n\n).

In [ ]:
from model import GPT, GPTConfig

# Load trained model
out_dir = 'out-aozora'
ckpt_path = f'{out_dir}/ckpt.pt'

if not os.path.exists(ckpt_path):
    print(f'ERROR: {ckpt_path} not found!')
    print('Training may have failed (OOM?). Check the training cell output above.')
    print('Try reducing batch_size further or max_iters.')
    raise FileNotFoundError(ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model_args = ckpt['model_args']
model_args['use_turboquant'] = True
model_args['turboquant_bits'] = 3

config = GPTConfig(**model_args)
model = GPT(config)

state_dict = ckpt['model']
for k in list(state_dict.keys()):
    if k.startswith('_orig_mod.'):
        state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

# Load tokenizer
with open('data/aozora/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f'Model loaded: {model.get_num_params()/1e6:.2f}M parameters')
print(f'Config: {config.n_layer}L, {config.n_head}H, {config.n_embd}D, block_size={config.block_size}')
print(f'Vocab size: {config.vocab_size}')

In [ ]:
def generate_with_stop(prompt, max_tokens=1024, use_turboquant=False,
                       temperature=0.9, top_k=200,
                       stop_sequences=None, seed=42):
    """
    Generate text with stop sequence support.

    Args:
        stop_sequences: list of strings to stop at (e.g., ['。', '\n\n'])
                        If None, defaults to ['。']
        temperature: higher values = more creative/diverse (default 0.9)
    """
    if stop_sequences is None:
        stop_sequences = ['。', '\n\n']

    start_ids = encode(prompt)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

    torch.manual_seed(seed)
    t0 = time.time()
    with torch.no_grad():
        y = model.generate(x, max_tokens, temperature=temperature, top_k=top_k,
                           use_turboquant=use_turboquant)
    elapsed = time.time() - t0

    full_text = decode(y[0].tolist())
    generated = full_text[len(prompt):]
    n_generated = len(generated)

    # Find first stop sequence
    stop_pos = len(generated)
    matched_stop = None
    for stop in stop_sequences:
        idx = generated.find(stop)
        if idx >= 0 and idx + len(stop) < stop_pos:
            stop_pos = idx + len(stop)
            matched_stop = repr(stop)

    trimmed = generated[:stop_pos]
    tok_per_sec = n_generated / elapsed

    return {
        'prompt': prompt,
        'generated': trimmed,
        'full_generated': generated,
        'stopped_by': matched_stop or f'max_tokens ({max_tokens})',
        'elapsed': elapsed,
        'tok_per_sec': tok_per_sec,
        'n_tokens': n_generated,
    }

In [ ]:
# Generate and compare: Standard vs TurboQuant
prompts = [
    'ジョバンニは、',
    'カムパネルラが、',
    '夜の軽便鉄道は、',
]

max_tokens = 1024

for prompt in prompts:
    print('=' * 60)
    print(f'Prompt: 「{prompt}」')
    print('=' * 60)

    # Standard
    r_std = generate_with_stop(prompt, max_tokens=max_tokens, use_turboquant=False)
    print(f'\n--- Standard ({r_std["tok_per_sec"]:.0f} tok/s) ---')
    print(f'{prompt}{r_std["generated"]}')
    print(f'  [Stopped by: {r_std["stopped_by"]}]')

    # TurboQuant
    r_tq = generate_with_stop(prompt, max_tokens=max_tokens, use_turboquant=True)
    print(f'\n--- TurboQuant 3-bit ({r_tq["tok_per_sec"]:.0f} tok/s) ---')
    print(f'{prompt}{r_tq["generated"]}')
    print(f'  [Stopped by: {r_tq["stopped_by"]}]')
    print()

In [ ]:
# Generate longer passages (stop at paragraph boundary)
prompt = 'ジョバンニは、'

print('=== Long generation (stop at paragraph break) ===')
print(f'Prompt: 「{prompt}」\n')

r = generate_with_stop(
    prompt,
    max_tokens=1024,
    use_turboquant=True,
    stop_sequences=['\n\n'],
    temperature=0.7,
)
print(f'{prompt}{r["generated"]}')
print(f'\n[{r["n_tokens"]} tokens | {r["elapsed"]:.2f}s | {r["tok_per_sec"]:.0f} tok/s | Stopped by: {r["stopped_by"]}]')

## 5. Quality Analysis: Bit-width Comparison

Compare text quality at different TurboQuant bit-widths.

In [ ]:
prompt = 'カムパネルラは、'

for bits in [None, 4, 3, 2]:
    label = f'TQ-{bits}bit' if bits else 'Standard'
    use_tq = bits is not None

    if use_tq:
        for block in model.transformer.h:
            block.attn.turboquant_bits = bits
            block.attn._tq_compressor = None

    r = generate_with_stop(prompt, max_tokens=512, use_turboquant=use_tq,
                           stop_sequences=['。'])

    print(f'--- {label} ({r["tok_per_sec"]:.0f} tok/s) ---')
    print(f'{prompt}{r["generated"]}')
    print(f'  [Stopped by: {r["stopped_by"]}]')
    print()

# Reset to 3-bit
for block in model.transformer.h:
    block.attn.turboquant_bits = 3
    block.attn._tq_compressor = None

## 6. Long Context Compression Benchmarks

TurboQuant's memory savings are most significant with longer sequences.
Compare compression at different context lengths.

In [ ]:
from turboquant import TurboQuantKVCompressor

head_dim = config.n_embd // config.n_head
n_heads = config.n_head
n_layers = config.n_layer

print(f'Model: {n_layers} layers, {n_heads} heads, head_dim={head_dim}')
print(f'Block size: {config.block_size}\n')

print(f'{"Seq Len":>8} | {"Bits":>4} | {"FP16 KV":>10} | {"TQ KV":>10} | {"Ratio":>6} | {"Saved":>8}')
print('-' * 65)

for T in [256, 512, 1024]:
    for bits in [3, 4]:
        comp = TurboQuantKVCompressor(head_dim, bits=bits)
        info = comp.memory_savings_info(T, n_heads)
        fp16_kb = info['fp16_bits'] / 8 / 1024 * n_layers * 2
        tq_kb = info['compressed_bits'] / 8 / 1024 * n_layers * 2
        saved_kb = fp16_kb - tq_kb
        print(f'{T:>8} | {bits:>4} | {fp16_kb:>8.1f} KB | {tq_kb:>8.1f} KB | {info["compression_ratio"]:>5.1f}x | {saved_kb:>6.1f} KB')

In [ ]:
import matplotlib.pyplot as plt

# Speed benchmark at different sequence lengths
bench_device = device
D = head_dim
H = n_heads

seq_lens = [128, 256, 512, 1024]
results = {}

for T in seq_lens:
    B = 2
    k = torch.randn(B, H, T, D, device=bench_device)
    v = torch.randn(B, H, T, D, device=bench_device)
    q = torch.randn(B, H, T, D, device=bench_device)

    comp = TurboQuantKVCompressor(D, bits=3, device=bench_device)

    ck = comp.compress_keys(k)
    cv = comp.compress_values(v)
    _ = comp.attention_scores(q, ck)
    _ = comp.decompress_values(cv)
    if bench_device == 'cuda':
        torch.cuda.synchronize()

    n_iter = 20

    t0 = time.time()
    for _ in range(n_iter):
        ck = comp.compress_keys(k)
        cv = comp.compress_values(v)
        _ = comp.attention_scores(q, ck)
        _ = comp.decompress_values(cv)
    if bench_device == 'cuda':
        torch.cuda.synchronize()
    t_tq = (time.time() - t0) / n_iter * 1000

    t0 = time.time()
    for _ in range(n_iter):
        s = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / math.sqrt(D))
        _ = torch.matmul(torch.softmax(s, dim=-1), v)
    if bench_device == 'cuda':
        torch.cuda.synchronize()
    t_std = (time.time() - t0) / n_iter * 1000

    info = comp.memory_savings_info(T, H)
    results[T] = {'t_tq': t_tq, 't_std': t_std, 'ratio': info['compression_ratio']}
    print(f'T={T:>5}: TQ={t_tq:.1f}ms, Std={t_std:.1f}ms, Compression={info["compression_ratio"]:.1f}x')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ts = list(results.keys())
tq_times = [results[t]['t_tq'] for t in ts]
std_times = [results[t]['t_std'] for t in ts]
ratios = [results[t]['ratio'] for t in ts]

ax1.plot(ts, std_times, 'b-o', label='Standard', linewidth=2)
ax1.plot(ts, tq_times, 'r-o', label='TurboQuant 3-bit', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Attention Computation Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

fp16_mem = [t * D * H * 2 * 16 / 8 / 1024 for t in ts]
tq_mem = [fp16_mem[i] / ratios[i] for i in range(len(ts))]

ax2.bar([str(t) for t in ts], fp16_mem, alpha=0.7, label='FP16', color='steelblue')
ax2.bar([str(t) for t in ts], tq_mem, alpha=0.7, label='TQ 3-bit', color='coral')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('KV Cache Size (KB)')
ax2.set_title('KV Cache Memory per Layer')
ax2.legend()

plt.suptitle('TurboQuant Benefits Scale with Sequence Length', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

### Japanese Text Generation with TurboQuant

| Setting | Compression | Quality |
|---------|-------------|--------|
| Standard (FP16) | 1.0x | Baseline |
| TurboQuant 4-bit | ~3.8x | Imperceptible difference |
| TurboQuant 3-bit | ~4.9x | Near-lossless |
| TurboQuant 2-bit | ~6.5x | Noticeable degradation |

### Key Observations

- **block_size=1024** enables longer, more coherent Japanese passages
- **Stop sequences** (。, \n\n) produce natural sentence/paragraph endings
- **Character-level tokenization** works well for Japanese (vocab ~2000-3000)
- **TurboQuant 3-bit** is the sweet spot: ~5x compression with minimal quality loss
- **Memory savings scale linearly** with sequence length: longer contexts benefit more